# Qrylov

## Setup

In [ ]:
from typing import List, Tuple
import warnings

import scienceplots
import matplotlib.pyplot as plt
plt.style.use("science")
import matplotlib.colors as mcolors
import numpy as np
import scipy as sp
import scipy.linalg

import openfermion as of
from openfermionpyscf import run_pyscf
import cirq
import qiskit
import qiskit.qasm2
import qiskit.quantum_info
import quimb.tensor as qtn

try:
    from tensor_network_common import pauli_sum_to_mpo
except ImportError:
    pauli_sum_to_mpo = None

In [ ]:
def cirq_pauli_sum_to_qiskit_pauli_op(pauli_sum, qubits):
    """Pauli labels follow the tensor-factor order supplied in `qubits`.

    The resulting Qiskit dense matrix uses that same tensor-factor order.
    Qiskit's numbered wires are reversed when exporting to Quimb's site order.
    """
    cirq_pauli_to_str = {cirq.X: "X", cirq.Y: "Y", cirq.Z: "Z"}
    terms, coeffs = [], []
    for term in pauli_sum:
        label = "".join(cirq_pauli_to_str[term[q]] if q in term else "I"
                        for q in qubits)
        if abs(term.coefficient.imag) > 1e-12:
            raise ValueError("The Pauli Hamiltonian has non-real coefficients.")
        terms.append(label)
        coeffs.append(term.coefficient.real)
    return qiskit.quantum_info.SparsePauliOp(terms, coeffs)

## Parameters

In [ ]:
# Problem to use.
problem: str = "hchain"  # Options: "hchain", "hubbard"
xdim: int = 2
ydim: int = 3
natoms: int = 16
bond_distance = 1.0 * 0.7414
hamiltonian_threshold = None

# Initial state: normalized (ground_state + random_state / alpha).
seed: int = 1
alpha: float = 1.0

# Total number of basis vectors, including the initial state.
# An odd value is required for the symmetric unitary time grid.
subspace_dimension: int = 21
assert subspace_dimension >= 1 and subspace_dimension % 2 == 1

# Generalized-eigenproblem cutoffs act on eigenvalues of S.
# These are starting values for noiseless double-precision matrix elements,
# not universal thresholds for sampled or compressed data.
threshold: float = 1e-10       # Absolute overlap eigenvalue threshold.
overlap_rtol: float = 1e-12   # Additional threshold relative to lambda_max(S).

# Explicit-state SVD cutoff acts on singular values of V, NOT eigenvalues of S.
# sigma(V)^2 = lambda(S). This much smaller state-level cutoff gives an ideal
# subspace baseline; it deliberately need not match the GEVP's noise cutoff.
svd_rtol: float = 1e-12
dense_solver: str = "svd"    # "svd" or "gevp"; both answers are recorded.
assert dense_solver in ("svd", "gevp")

max_mpo_bond: int = 10
hermitize_mpo: bool = True  # After compression, do (MPO + MPO^\dagger) / 2
max_mps_bond: int = 20
mps_cutoff: float = 0.0
MAX_NQUBITS_FOR_MATRIX_ALGEBRA: int = 12


In [ ]:
def checked_hermitian(a, name="matrix", rtol=1e-10):
    """Validate before removing roundoff asymmetry; never conceal a bad MPO."""
    a = np.asarray(a, dtype=np.complex128)
    if a.ndim != 2 or a.shape[0] != a.shape[1] or not np.all(np.isfinite(a)):
        raise ValueError(f"{name} must be a finite square matrix.")
    defect = sp.linalg.norm(a - a.conj().T, "fro")
    scale = max(sp.linalg.norm(a, "fro"), np.finfo(float).tiny)
    if defect > rtol * scale:
        raise ValueError(f"{name} is not Hermitian: relative defect {defect/scale:.3e}. "
                         "Fix the Hamiltonian/contractions before solving.")
    return 0.5 * (a + a.conj().T)


def dense_vector(v):
    a = v.to_dense() if hasattr(v, "to_dense") else v
    return np.asarray(a, dtype=np.complex128).reshape(-1)


def fill_h_and_s_matrices(vectors, mpo, verbose=False):
    """Compute the actual projection. Do not manufacture a Hermitian triangle."""
    if isinstance(mpo, np.ndarray):
        operator = checked_hermitian(mpo, "Hamiltonian")
        V = np.column_stack([dense_vector(v) for v in vectors])
        h = V.conj().T @ (operator @ V)
        s = V.conj().T @ V
    else:
        # No compression when evaluating matrix elements. Approximating the
        # basis changes the subspace; approximating entries separately can
        # destroy the consistency needed for the variational principle.
        applied = [mpo.apply(v, compress=False) for v in vectors]
        h = np.array([[complex(vi.H @ hvj) for hvj in applied]
                      for vi in vectors], dtype=np.complex128)
        s = np.array([[complex(vi.H @ vj) for vj in vectors]
                      for vi in vectors], dtype=np.complex128)
    h = checked_hermitian(h, "projected Hamiltonian")
    s = checked_hermitian(s, "overlap matrix")
    if verbose:
        print(f"Constructed {len(vectors)} x {len(vectors)} projected matrices.")
    return h, s


def solve_regularized_gen_eig(h, s, threshold=1e-10, rtol=1e-12,
                              return_info=False):
    """Canonical orthogonalization, with only reliable positive S eigenvalues."""
    if threshold < 0 or rtol < 0:
        raise ValueError("Use nonnegative overlap thresholds.")
    h = checked_hermitian(h, "projected Hamiltonian")
    s = checked_hermitian(s, "overlap matrix")
    s_vals, s_vecs = sp.linalg.eigh(s)
    if s_vals[-1] <= 0:
        raise ValueError("The overlap matrix has no positive eigenvalues.")
    cutoff = max(threshold, rtol * s_vals[-1],
                 len(s) * np.finfo(float).eps * s_vals[-1])
    if s_vals[0] < -10 * cutoff:
        raise ValueError("S is substantially indefinite; check matrix elements.")
    keep = s_vals > cutoff
    if not np.any(keep):
        raise ValueError("No overlap eigenvalues survive the threshold.")
    W = s_vecs[:, keep] / np.sqrt(s_vals[keep])[None, :]
    h_orth = W.conj().T @ h @ W
    h_orth = 0.5 * (h_orth + h_orth.conj().T)
    energy = float(sp.linalg.eigvalsh(h_orth)[0])
    info = dict(gevp_rank=int(keep.sum()), s_min=float(s_vals[0]),
                s_max=float(s_vals[-1]), s_cutoff=float(cutoff),
                s_condition=(float(s_vals[-1] / s_vals[0])
                             if s_vals[0] > 0 else np.inf))
    return (energy, info) if return_info else energy


def solve_statevector_ritz(vectors, matrix, rtol=1e-12, return_info=False):
    """Stable noiseless baseline using available state amplitudes.

    This classical benchmark does not model finite-shot matrix estimation.
    Re-thresholding at each dimension can make retained spaces non-nested.
    """
    if rtol < 0:
        raise ValueError("SVD rtol must be nonnegative.")
    matrix = checked_hermitian(matrix, "Hamiltonian")
    V = np.column_stack([dense_vector(v) for v in vectors])
    Q, singular_values, _ = sp.linalg.svd(V, full_matrices=False)
    if singular_values[0] == 0:
        raise ValueError("All basis vectors are zero.")
    keep = singular_values > rtol * singular_values[0]
    Q = Q[:, keep]
    projected = Q.conj().T @ (matrix @ Q)
    projected = 0.5 * (projected + projected.conj().T)
    values, coeffs = sp.linalg.eigh(projected)
    state = Q @ coeffs[:, 0]
    energy = float(values[0])
    info = dict(svd_rank=int(keep.sum()),
                sigma_min=float(singular_values[-1]),
                sigma_max=float(singular_values[0]),
                ritz_residual=float(sp.linalg.norm(matrix @ state - energy * state)))
    return (energy, info) if return_info else energy


def evaluate_subspace(vectors, operator):
    """Record both the explicit-state and matrix-element answers when feasible."""
    h, s = fill_h_and_s_matrices(vectors, operator)
    eg, info = solve_regularized_gen_eig(h, s, threshold=threshold,
                                         rtol=overlap_rtol, return_info=True)
    info.update(d=len(vectors), gevp_energy=eg)
    if isinstance(operator, np.ndarray):
        es, si = solve_statevector_ritz(vectors, operator, rtol=svd_rtol,
                                        return_info=True)
        info.update(si, svd_energy=es)
        energy = es if dense_solver == "svd" else eg
        if es < energy_exact - 1e-9:
            raise RuntimeError("The statevector Ritz value violates the ground-energy bound.")
        if eg < energy_exact - 1e-8:
            warnings.warn("GEVP violates the variational bound; increase the cutoff "
                          "or investigate inaccurate matrix elements.")
    else:
        energy = eg
    info["energy"] = energy
    print(f"d={len(vectors):2d}, E={energy:.12f}, min eig(S)={info['s_min']:.3e}, "
          f"GEVP rank={info['gevp_rank']}")
    return energy, info


## Problem definition

In [ ]:
if problem == "hchain":
    geometry = [("H", (0, 0, i * bond_distance)) for i in range(natoms)]
    molecule = of.MolecularData(geometry, basis="sto-3g", multiplicity=1, charge=natoms % 2)

    molecule = run_pyscf(molecule, run_scf=True, run_mp2=True, verbose=True)

    hamiltonian = of.get_fermion_operator(molecule.get_molecular_hamiltonian())
    hamiltonian = of.jordan_wigner(hamiltonian)

elif problem == "hubbard":
    hamiltonian = of.jordan_wigner(
        of.hamiltonians.fermi_hubbard(
            xdim,
            ydim,
            1.0,
            1.0,
            spinless=True
        )
    )

else:
    raise ValueError(f"Unknown problem {problem}.")

In [ ]:
if hamiltonian_threshold is not None:
    hamiltonian.compress(hamiltonian_threshold)

In [ ]:
nqubits = of.utils.count_qubits(hamiltonian)
nterms = len(hamiltonian.terms)
print(f"Hamiltonian acts on {nqubits} qubit(s) and has {nterms} Pauli terms.")

## Hamiltonian audit and reference energy


In [ ]:
hamiltonian_cirq = of.qubit_operator_to_pauli_sum(hamiltonian)
qubits = tuple(cirq.LineQubit.range(nqubits))
hamiltonian_qiskit = cirq_pauli_sum_to_qiskit_pauli_op(hamiltonian_cirq, qubits)
dense_mode = nqubits <= MAX_NQUBITS_FOR_MATRIX_ALGEBRA
matrix = None
mpo = None

if dense_mode:
    # Independent, uncompressed source of truth. Do not use mpo.to_dense().
    matrix = checked_hermitian(
        of.get_sparse_operator(hamiltonian, n_qubits=nqubits).toarray(),
        "uncompressed Pauli Hamiltonian")
    h_qiskit = np.asarray(hamiltonian_qiskit.to_matrix(), dtype=np.complex128)
    if not np.allclose(matrix, h_qiskit, rtol=1e-12, atol=1e-12):
        raise ValueError("OpenFermion/Qiskit tensor-factor ordering mismatch.")
    print("Uncompressed OpenFermion and Qiskit matrices agree.")

if pauli_sum_to_mpo is not None:
    mpo = pauli_sum_to_mpo(hamiltonian_cirq, qubits,
                          max_bond=max_mpo_bond, verbose=True)
    if hermitize_mpo:
        # mpo = (mpo + mpo.H) / 2
        mpo_dagger = mpo.conj().partial_transpose(range(mpo.L))
        mpo = 0.5 * (mpo + mpo_dagger)

    mpo.show()
    if dense_mode:
        old_matrix = np.asarray(mpo.to_dense(), dtype=np.complex128)
        if not np.all(np.isfinite(old_matrix)):
            warnings.warn("The original compressed MPO contains nonfinite entries.")
        else:
            print("Original MPO Hermiticity defect (Frobenius):",
                  sp.linalg.norm(old_matrix - old_matrix.conj().T, "fro"))
            print("Original MPO error versus Pauli Hamiltonian (Frobenius):",
                  sp.linalg.norm(old_matrix - matrix, "fro"))
        print("This MPO is audited only; the dense benchmark uses the exact Pauli matrix.")
elif not dense_mode:
    raise ImportError("The large-system branch requires tensor_network_common.py.")
else:
    print("Optional MPO helper unavailable; proceeding with the exact dense benchmark.")

operator = matrix if dense_mode else mpo

In [ ]:
if dense_mode:
    evals, evecs = sp.linalg.eigh(matrix)
    energy_exact = float(evals[0])
    # Fix the arbitrary ground-vector phase for reproducibility.
    g = evecs[:, 0].copy()
    g *= np.exp(-1j * np.angle(g[np.argmax(np.abs(g))]))
    ground_state = qtn.MatrixProductState.from_dense(g, dims=2, cutoff=0.0)
    reference_label = "Exact Pauli Hamiltonian"
    print("Exact energy:", energy_exact)
else:
    dmrg = qtn.DMRG2(mpo, bond_dims=[2 ** (i + 1)
                     for i in range(int(np.log2(max_mps_bond)))])
    dmrg.solve(verbosity=1)
    energy_exact = float(np.real(dmrg.energy))  # Historical variable name only.
    ground_state = dmrg.state
    reference_label = "DMRG reference (approximate MPO)"
    warnings.warn("The reference is approximate. Check MPO Hermiticity and converge "
                  "both MPO and DMRG accuracy before interpreting small errors.")
    print("Approximate reference energy:", energy_exact)
    print("Imaginary part of DMRG energy:", np.imag(dmrg.energy))
ground_state.show()

## Initial state

In [ ]:
bvec = ground_state + qtn.MPS_rand_state(nqubits, bond_dim=2, seed=seed) / alpha
bvec /= bvec.norm()
overlap = float(np.abs(ground_state.H @ bvec) ** 2)
print("Initial ground-state overlap:", overlap)
if dense_mode:
    psi0 = dense_vector(bvec)
    psi0 /= sp.linalg.norm(psi0)

## Krylov with $H$

In [ ]:
vectors = [psi0.copy() if dense_mode else bvec.copy()]
for i in range(1, subspace_dimension):
    if dense_mode:
        new = matrix @ vectors[-1]
        new_norm = sp.linalg.norm(new)
    else:
        new = mpo.apply(vectors[-1], compress=True,
                        max_bond=max_mps_bond, cutoff=mps_cutoff)
        new_norm = new.norm()
    if new_norm == 0:
        print("Power sequence terminated at an exact zero vector.")
        break
    new /= new_norm
    vectors.append(new)
print("Number of power basis vectors:", len(vectors))


In [ ]:
h_power, s_power = fill_h_and_s_matrices(vectors, operator, verbose=True)

In [ ]:
krylov_dvals, krylov_evals, krylov_diagnostics = [], [], []
for d in range(1, len(vectors) + 1):
    energy, info = evaluate_subspace(vectors[:d], operator)
    krylov_dvals.append(d)
    krylov_evals.append(energy)
    krylov_diagnostics.append(info)

In [ ]:
plt.plot(krylov_dvals, krylov_evals, "--o", label="Krylov with $H$")
plt.axhline(energy_exact, ls="--", color="black", label=reference_label)
plt.xlabel("Number of basis vectors $d$")
plt.ylabel("Energy (Ha)")
plt.legend();

### Spectral analysis

In [ ]:
if dense_mode:
    print("Ground-state energy:", evals[0])
    print("Maximum eigenvalue:", evals[-1])
    print("Spectral bandwidth:", evals[-1] - evals[0])
    print("Pauli coefficient L1 norm:", np.sum(np.abs(hamiltonian_qiskit.coeffs)))
    print("Hamiltonian condition number:", np.linalg.cond(matrix))
    print("Krylov stability is governed by the BASIS/OVERLAP conditioning, not this number.")
    plt.plot(evals, "o")
    plt.ylabel("Energy (Ha)")


## Krylov with $U = \exp(-i H t)$

In [ ]:
if dense_mode:
    bandwidth = evals[-1] - evals[0]
else:
    # Emax-Emin <= 2*sum(abs(c_P)) over nonidentity Pauli terms.
    bandwidth = 2 * sum(abs(c) for label, c in hamiltonian_qiskit.to_list()
                        if label != "I" * nqubits)
if bandwidth <= 0:
    raise ValueError("The Hamiltonian has no nontrivial spectral bandwidth.")
dt = np.pi / bandwidth
print("dt =", dt)


In [ ]:
krylov_u_dvals, krylov_u_evals, krylov_u_diagnostics = [], [], []
if dense_mode:
    midpoint = (subspace_dimension - 1) // 2
    time_indices = np.arange(-midpoint, midpoint + 1)
    spectral_coeffs = evecs.conj().T @ psi0
    vectors_u = [evecs @ (np.exp(-1j * evals * k * dt) * spectral_coeffs)
                 for k in time_indices]
    print("Exact-unitary time indices:", time_indices)
    print("Maximum state norm defect:",
          max(abs(sp.linalg.norm(v) - 1) for v in vectors_u))
    for radius in range(midpoint + 1):
        selected = vectors_u[midpoint-radius:midpoint+radius+1]
        energy, info = evaluate_subspace(selected, matrix)
        krylov_u_dvals.append(len(selected))
        krylov_u_evals.append(energy)
        krylov_u_diagnostics.append(info)


In [ ]:
if dense_mode:
    plt.plot(krylov_dvals, krylov_evals, "--o", label="Krylov with $H$")
    plt.plot(krylov_u_dvals, krylov_u_evals, "--o", label="Krylov with $U$")
    plt.axhline(energy_exact, ls="--", color="black", label=reference_label)
    plt.xlabel("Number of basis vectors $d$")
    plt.ylabel("Energy (Ha)")
    plt.legend();


## Trotterized unitary Krylov

In [ ]:
ntrotter_values = [1, 2, 4, 8]
trotter_circuits = []
all_krylov_u_trotter_dvals = []
all_krylov_u_trotter_evals = []
all_krylov_u_trotter_diagnostics = []
all_vectors_u_trotter = []
midpoint = (subspace_dimension - 1) // 2

for ntrotter in ntrotter_values:
    print("Trotter repetitions per dt:", ntrotter)
    trotter_operation = qiskit.circuit.library.PauliEvolutionGate(
        hamiltonian_qiskit, time=dt,
        synthesis=qiskit.synthesis.LieTrotter(reps=ntrotter))
    circuit = qiskit.QuantumCircuit(nqubits)
    circuit.append(trotter_operation, circuit.qubits)
    # Synthesize first: evolving the unsynthesized PauliEvolutionGate may use
    # its exact matrix and accidentally bypass the intended product formula.
    circuit = qiskit.transpile(circuit, basis_gates=["u3", "cx"],
                               seed_transpiler=seed)
    print("Gate counts per dt:", circuit.count_ops())
    trotter_circuit = circuit.reverse_bits()  # Quimb numbered-site convention.
    trotter_circuits.append(trotter_circuit)

    if dense_mode:
        # The original Qiskit circuit's dense tensor order matches matrix,
        # checked above. Do NOT reverse its bits in this branch.
        inverse_circuit = circuit.inverse()
        forward_vectors, reverse_vectors = [psi0.copy()], [psi0.copy()]
        for _ in range(midpoint):
            forward_vectors.append(qiskit.quantum_info.Statevector(
                forward_vectors[-1]).evolve(circuit).data)
            reverse_vectors.append(qiskit.quantum_info.Statevector(
                reverse_vectors[-1]).evolve(inverse_circuit).data)
    else:
        forward_qasm = qiskit.qasm2.dumps(trotter_circuit)
        reverse_qasm = qiskit.qasm2.dumps(trotter_circuit.inverse())
        forward_vectors, reverse_vectors = [bvec.copy()], [bvec.copy()]
        for _ in range(midpoint):
            forward_vectors.append(qtn.CircuitMPS.from_openqasm2_str(
                forward_qasm, psi0=forward_vectors[-1], progbar=False,
                max_bond=max_mps_bond, cutoff=mps_cutoff).psi)
            reverse_vectors.append(qtn.CircuitMPS.from_openqasm2_str(
                reverse_qasm, psi0=reverse_vectors[-1], progbar=False,
                max_bond=max_mps_bond, cutoff=mps_cutoff).psi)

    # Repeated powers of ONE fixed approximate unitary; the backward branch
    # uses its actual inverse. This yields nested symmetric subspaces.
    vectors_u_trotter = list(reversed(reverse_vectors[1:])) + forward_vectors
    all_vectors_u_trotter.append(vectors_u_trotter)
    if dense_mode:
        print("Maximum state norm defect:",
              max(abs(sp.linalg.norm(v)-1) for v in vectors_u_trotter))

    dvals, energies, diagnostics = [], [], []
    for radius in range(midpoint + 1):
        selected = vectors_u_trotter[midpoint-radius:midpoint+radius+1]
        energy, info = evaluate_subspace(selected, operator)
        dvals.append(len(selected))
        energies.append(energy)
        diagnostics.append(info)
    all_krylov_u_trotter_dvals.append(dvals)
    all_krylov_u_trotter_evals.append(energies)
    all_krylov_u_trotter_diagnostics.append(diagnostics)


In [ ]:
kwargs = {"alpha": 0.667, "mec": "black", "ms": 6}
fig, ax = plt.subplots()

def plot_error(dvals, energies, **plot_kwargs):
    signed = np.asarray(energies) - energy_exact
    if dense_mode and np.any(signed < -1e-8):
        warnings.warn("A plotted estimate lies below E0. Inspect signed errors below.")
    # A floor is only for plotting exact zeros on a logarithmic axis.
    ax.plot(dvals, np.maximum(np.abs(signed), 1e-16), "--s", **plot_kwargs)

plot_error(krylov_dvals, krylov_evals, label="$H$", color="black", **kwargs)
u_color = "tab:red"
if dense_mode:
    plot_error(krylov_u_dvals, krylov_u_evals, label="$U$",
               color=u_color, zorder=10, **kwargs)

cmap = mcolors.LinearSegmentedColormap.from_list("trotter_to_u", ["#bcd8ee", u_color])
log_ntrotter = np.log2(ntrotter_values)
norm = mcolors.Normalize(vmin=log_ntrotter.min(), vmax=log_ntrotter.max())
for ntrotter, dvals, energies in zip(ntrotter_values,
                                    all_krylov_u_trotter_dvals,
                                    all_krylov_u_trotter_evals):
    plot_error(dvals, energies, color=cmap(norm(np.log2(ntrotter))), **kwargs)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label("Trotter steps per $\\Delta t$")
cbar.set_ticks(log_ntrotter)
cbar.set_ticklabels([str(n) for n in ntrotter_values])
ax.axhline(1e-3, ls="--", color="gray", label="1 mHa")
ax.set_xlabel("Number of basis vectors $d$")
ax.set_ylabel("Absolute energy error (Ha)")
ax.set_yscale("log")
ax.legend(fontsize=10)
fig.tight_layout()
fig.savefig(f"krylov_{problem}_nqubits_{nqubits}_overlap_{overlap}_threshold_{threshold}_max_trotter_{max(ntrotter_values)}_dt_{dt}_seed_{seed}.pdf")

## Resource analysis

In [ ]:
ngates_oneq, ngates_twoq, depths = [], [], []
for circuit in trotter_circuits:
    counts = circuit.count_ops()
    ngates_oneq.append(counts.get("u3", 0))
    ngates_twoq.append(counts.get("cx", 0))
    depths.append(circuit.depth())


In [ ]:
plt.plot(ntrotter_values, ngates_oneq, "--o", label="One-qubit gates")
plt.plot(ntrotter_values, ngates_twoq, "--o", label="Two-qubit gates")
plt.plot(ntrotter_values, depths, "--o", label="Circuit depth")

plt.xlabel("Number of Trotter steps")
plt.ylabel("Count")

plt.legend();

## Numerical diagnostics and interpretation

`S = V.conj().T @ V` squares the condition number of `V`. The SVD result below
is the noiseless explicit-state baseline. The GEVP result estimates what can
be recovered from the projected matrices at the chosen overlap threshold.
The two truncations differ: an overlap cutoff `tau` corresponds to a state
singular-value cutoff `sqrt(tau)`, not `tau`.

Use a common Hamiltonian, initial state, time step, and cutoff policy for all
methods. Inspect effective rank as well as number of prepared vectors. For
the large-system branch also converge MPO and MPS bond dimensions separately.

With exact projections and nested subspaces, each minimum energy is bounded
below by E0 and is nonincreasing with dimension. Thresholding can make the
retained subspaces non-nested, so small upward steps or plateaus may occur.
Trotterization changes the basis and the convergence rate, but does not itself
permit an energy below E0 when the projected Hamiltonian is evaluated exactly.
Convergence as the Trotter repetition count increases need not be monotone.

For a convergence-rate study, compare the SVD curves first; then vary the GEVP
cutoffs (e.g. 1e-8, 1e-10, 1e-12 for exact double-precision matrix elements) and
inspect sensitivity. Smaller dt can make neighboring states more dependent;
it is not automatically a stability fix. The power basis can also become
ill-conditioned at larger dimension.

References: [Epperly–Lin–Nakatsukasa](https://arxiv.org/abs/2110.07492),
[SciPy eigh](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.eigh.html).


In [ ]:
series = [("Power", krylov_diagnostics)]
if dense_mode:
    series.append(("Exact U", krylov_u_diagnostics))
series += [(f"Trotter r={r}", ds) for r, ds in
           zip(ntrotter_values, all_krylov_u_trotter_diagnostics)]
for name, diagnostics in series:
    print("\n" + name)
    print(" d    E-Eref (Ha)       min eig(S)     GEVP rank   SVD rank     GEVP-SVD (Ha)")
    for row in diagnostics:
        difference = row.get("gevp_energy", np.nan) - row.get("svd_energy", np.nan)
        print(f"{row['d']:2d}  {row['energy']-energy_exact: .6e}   {row['s_min']: .3e}"
              f"       {row['gevp_rank']:3d}         {row.get('svd_rank', 0):3d}"
              f"        {difference: .3e}")

if dense_mode:
    # This sweep isolates matrix-solve sensitivity using exactly the same
    # already-generated vectors. No time evolution or chemistry is rerun.
    final_bases = [("Power", vectors), ("Exact U", vectors_u)]
    final_bases += [(f"Trotter r={r}", vs)
                    for r, vs in zip(ntrotter_values, all_vectors_u_trotter)]
    print("\nFinal-dimension threshold sensitivity")
    for name, vs in final_bases:
        hh, ss = fill_h_and_s_matrices(vs, matrix)
        es = solve_statevector_ritz(vs, matrix, rtol=svd_rtol)
        print(f"\n{name}: SVD energy = {es:.12f}")
        for tau in (1e-8, 1e-10, 1e-12):
            eg, inf = solve_regularized_gen_eig(hh, ss, threshold=tau,
                                                rtol=0.0, return_info=True)
            print(f"  absolute cutoff={tau:.0e}, rank={inf['gevp_rank']}, "
                  f"E={eg:.12f}, E-E0={eg-energy_exact:.3e}")
